# 추론 실습

**Inference · 예측**

학습된 모델을 새로운 입력에 적용해 출력을 얻는 과정.

소재 분야에서 이해하기: 학습을 마친 모델로 새 조성의 물성을 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 학습과 추론의 분리

학습은 한 번, 추론은 반복해서 값싸게 수행합니다. 시간을 재서 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor

start = time.perf_counter()
model = RandomForestRegressor(n_estimators=400, random_state=0).fit(X, y)
train_seconds = time.perf_counter() - start

candidates = np.column_stack([rng.uniform(600, 900, 20000), rng.uniform(0.5, 8, 20000),
                              rng.uniform(0, 5, 20000), rng.normal(0, 1, 20000)])
start = time.perf_counter()
predictions = model.predict(candidates)
infer_seconds = time.perf_counter() - start
print('학습 %.2f초 / 후보 2만 건 추론 %.2f초' % (train_seconds, infer_seconds))
print('추론 1건당 %.2f 마이크로초' % (infer_seconds / len(candidates) * 1e6))

## 2. 스크리닝에 쓰기

추론이 싸기 때문에 후보를 대량으로 걸러낼 수 있습니다.

In [ ]:
top = np.argsort(-predictions)[:5]
for rank, index in enumerate(top, 1):
    print('%d위 예측 %.1f HV / 조건 %s' % (rank, predictions[index],
          np.round(candidates[index, :3], 2)))
print('\n예측 상위 후보는 검증 대상 목록이며 확정된 결과가 아닙니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#inference)을 여세요.